# 1. Setup

**Approach 2 · DINOv2 fine-tuning depth**

This study starts from A2 in `Approach_2_Matched_Selection.ipynb`. Compare frozen A2 with the last 2, 4 or 6 DINOv2 Transformer blocks unfrozen. Open in the team repository and use **Run All**. With `DEBUG=False`, train, select, refit and evaluate all four recipes. With `DEBUG=True`, run one short training-only epoch per recipe.

In [ ]:
DEBUG = False

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
BATCH_SIZE = 8 if DEBUG else 16
EPOCHS = 1 if DEBUG else 20
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
RUN_NAME = "approach2_a2_finetune_depth" + ("_debug" if DEBUG else "")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = OUTPUT_DIR / "development"
FINAL_DIR = OUTPUT_DIR / "final"
OFFICIAL_DIR = OUTPUT_DIR / "official" / "pair_identity_order"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID

HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.1, 0.1
GAMMAS = [step / 2 for step in range(25)]
RECIPES = {"A2_Frozen": 0, "A4_FT2": 2, "A4_FT4": 4, "A4_FT6": 6}

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Missing training metadata or images under {team.DATA_DIR}.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the team's 323/333 common files.")
print(f"{DEVICE} | FP32 | batch {BATCH_SIZE} | {EPOCHS} epochs | DEBUG={DEBUG}")


# 2. Data

Use the team's train/validation split from `train.csv`. Reserve ten complete, training-derived crop–disease combinations and exclude their images from development training. Use five for recipe/epoch/calibration selection and the other five for a check, alongside disjoint seen validation images. Each development run uses the same fit images and data augmentation. Full refit restores all the training rows.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")


def calibration_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development training."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Calibration needs both seen and proxy images in each group.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY, "optimizer": "AdamW", "schedule": "CosineAnnealingLR",
    "label_smoothing": SMOOTHING, "decorrelation_weight": DECOR_WEIGHT,
    "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE, "unfrozen_blocks": RECIPES,
    "calibration_grid": GAMMAS, "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}


# 3. Model

All four recipes use two linear crop and disease heads, cross-entropy, logit Decorrelation, Cross-crop SupCon and the team's A2 image augmentation. The model uses the same additive crop–disease score in all arms. The heads learn at `1e-4` and the unfrozen blocks at `1e-5`. The patch embedding and final normalization are frozen. The model keeps its forward modules in evaluation mode during training, with gradients enabled on the heads and the selected blocks.

In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


# 4. Development and full refit

Use the raw harmonic mean of seen and proxy-unseen Top-1 to select each recipe's checkpoint and the overall depth. Break ties by proxy Top‑1, seen Top‑1, earlier epoch and fewer unfrozen blocks. At each chosen epoch, select the recipe's static calibration penalty from the tune split, then report the independent check.

Refit each recipe from the original pretrained DINOv2 weights and fresh heads on every training image, for its selected epoch count. Save and hash each complete model and its scoring rules before accessing official test data.

In [ ]:
def make_model(depth, *, pretrained=True):
    model = MultiTaskNet(pretrained=pretrained)
    blocks = model.backbone.blocks
    if len(blocks) != 12 or depth not in (0, 2, 4, 6):
        raise ValueError("Expected DINOv2 ViT-B/14 with 12 blocks and a supported fine-tuning depth.")
    if depth:
        blocks[-depth:].requires_grad_(True)
    return model.to(DEVICE).eval()


def make_optimizer(model):
    groups = [{"params": list(model.crop_head.parameters()) + list(model.disease_head.parameters()), "lr": HEAD_LR}]
    tail = [param for param in model.backbone.parameters() if param.requires_grad]
    if tail:
        groups.append({"params": tail, "lr": BACKBONE_LR})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def make_loader(frame, transform, shuffle):
    return DataLoader(PlantDataset(frame, transform, "imagenet"), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=team.NUM_WORKERS, drop_last=False,
                      generator=torch.Generator().manual_seed(team.SEED))


def train_epoch(model, optimizer, criterion, loader, recipe, stage, epoch, epochs):
    loss_total, samples, valid_anchors = 0., 0, 0
    for step, (images, crop, disease, _) in enumerate(loader, 1):
        images, crop, disease = (batch.to(DEVICE) for batch in (images, crop, disease))
        crop_logits, disease_logits = model(images)
        crop_repr, disease_repr = center_logits(crop_logits), center_logits(disease_logits)
        con, anchors = cross_crop_supcon(disease_repr, crop, disease)
        loss = (criterion(crop_logits, crop) + criterion(disease_logits, disease)
                + DECOR_WEIGHT * decorrelation(crop_repr, disease_repr) + SUPCON_WEIGHT * con)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss in {recipe}.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * len(images)
        samples += len(images)
        valid_anchors += anchors.item()
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{recipe} {stage} epoch {epoch}/{epochs} · batch {step}/{len(loader)}", flush=True)
    if samples != len(loader.dataset):
        raise RuntimeError("The fit loader did not process all intended images.")
    return {"epoch": epoch, "images_trained": samples,
            "train_loss": loss_total / samples, "supcon_anchor_fraction": valid_anchors / samples}


@torch.no_grad()
def development_scores(model, loader):
    scores, labels = [], []
    for images, _, _, target in loader:
        scores.append(pair_scores(*model(images.to(DEVICE)), train_pairs).cpu())
        labels.append(target)
    return torch.cat(scores), torch.cat(labels)


def calibration_metrics(seen, proxy, fit_mask, gamma):
    seen_scores, y_seen = seen
    proxy_scores, y_proxy = proxy
    seen_scores = seen_scores - gamma * fit_mask
    proxy_scores = proxy_scores - gamma * fit_mask
    seen_top1 = top_k_accuracy(seen_scores, y_seen, 1)
    proxy_top1 = top_k_accuracy(proxy_scores, y_proxy, 1)
    return {"gamma": gamma, "seen_top1": seen_top1, "proxy_top1": proxy_top1,
            "seen_top5": top_k_accuracy(seen_scores, y_seen, 5),
            "proxy_top5": top_k_accuracy(proxy_scores, y_proxy, 5),
            "harmonic_top1": harmonic_mean(seen_top1, proxy_top1)}


def selection_key(row):
    return row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"]


def snapshot_trainable(model):
    return {name: param.detach().cpu().clone()
            for name, param in model.named_parameters() if param.requires_grad}


@torch.no_grad()
def restore_trainable(model, state):
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.copy_(state[name])


def develop_recipe(recipe, depth, fit, fit_transform, groups, fit_mask):
    team.set_seed()
    model = make_model(depth)
    fit_loader = make_loader(fit, fit_transform, shuffle=True)
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    history, best, best_weights, best_scores = [], None, None, None
    for epoch in range(1, EPOCHS + 1):
        row = {"recipe": recipe, "unfrozen_blocks": depth,
               **train_epoch(model, optimizer, criterion, fit_loader, recipe, "development", epoch, EPOCHS)}
        scheduler.step()
        if not DEBUG:
            tune = [development_scores(model, groups[key]) for key in ("tune_seen", "tune_proxy")]
            row.update(calibration_metrics(*tune, fit_mask, 0.))
            if best is None or selection_key(row) > selection_key(best):
                best = row.copy()
                best_weights = snapshot_trainable(model)
                best_scores = tune
        history.append(row)
        print(row, flush=True)

    output = DEVELOPMENT_DIR / recipe
    output.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(output / "training_history.csv", index=False)
    if DEBUG:
        return None

    grid = [calibration_metrics(*best_scores, fit_mask, gamma) for gamma in GAMMAS]
    gamma = max(grid, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                       row["seen_top1"], -row["gamma"]))["gamma"]
    restore_trainable(model, best_weights)
    check = [development_scores(model, groups[key]) for key in ("check_seen", "check_proxy")]
    check_rows = [{"recipe": recipe, "scoring": rule, **calibration_metrics(*check, fit_mask, penalty)}
                  for rule, penalty in (("raw", 0.), ("calibrated", gamma))]
    pd.DataFrame(grid).to_csv(output / "calibration_tune_grid.csv", index=False)
    pd.DataFrame(check_rows).to_csv(output / "calibration_check.csv", index=False)
    return {"selection": best, "gamma": gamma, "independent_check": check_rows,
            "trainable_parameter_count": sum(p.numel() for p in model.parameters() if p.requires_grad)}


def run_development():
    team.set_seed()
    train, val, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Development loaders must use the training CSV only.")
    fit = train.dataset.frame
    fit_mask, frames, groups = None, {}, {}
    if not DEBUG:
        fit_mask, frames = calibration_split(fit, val.dataset.frame)
        allowed = fit_mask.nonzero(as_tuple=True)[0].tolist()
        fit = fit.loc[fit.pair_row.isin(allowed)]
        groups = {key: make_loader(frame, image_transform, shuffle=False) for key, frame in frames.items()}

    selections = {recipe: develop_recipe(recipe, depth, fit, train.dataset.transform, groups, fit_mask)
                  for recipe, depth in RECIPES.items()}
    if DEBUG:
        return {}
    # Independent check results are reported only. The winner uses raw tune metrics.
    chosen = max(selections, key=lambda recipe: (*selection_key(selections[recipe]["selection"]), -RECIPES[recipe]))
    plan = {"recipe_selected_on_tune": chosen, "recipes": selections,
            "tune_pair_rows": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
            "check_pair_rows": sorted(frames["check_proxy"].pair_row.unique().tolist())}
    atomic_json(DEVELOPMENT_DIR / "selection_plan.json", plan)
    pd.DataFrame([{**value["selection"], "locked_gamma": value["gamma"], "selected_on_tune": recipe == chosen}
                  for recipe, value in selections.items()]).to_csv(
        DEVELOPMENT_DIR / "selected_models.csv", index=False)
    display(pd.read_csv(DEVELOPMENT_DIR / "selected_models.csv"))
    return plan


plan = run_development()


In [ ]:
def read_final_lock(recipe):
    path = FINAL_DIR / recipe / "selection_lock.json"
    checkpoint = CHECKPOINT_DIR / recipe / "model.pt"
    lock = json.loads(path.read_text(encoding="utf-8"))
    if (lock["settings"] != settings or lock["recipe"] != recipe
            or lock["selection_plan"] != plan or lock["seen_pairs"] != train_pairs.tolist()
            or lock["checkpoint_sha256"] != sha256_file(checkpoint)):
        raise RuntimeError(f"Final checkpoint or plan changed for {recipe}.")
    return lock


def refit_recipe(recipe, depth, selection):
    team.set_seed()
    model = make_model(depth)
    train_transform, _ = get_transforms("imagenet")
    loader = make_loader(train_full, train_transform, shuffle=True)
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    epochs = selection["selection"]["epoch"]
    if not 1 <= epochs <= EPOCHS:
        raise ValueError(f"Invalid selected epoch for {recipe}.")
    history = []
    for epoch in range(1, epochs + 1):
        row = train_epoch(model, optimizer, criterion, loader, recipe, "full refit", epoch, epochs)
        scheduler.step()
        history.append(row)
        print(row, flush=True)

    checkpoint = CHECKPOINT_DIR / recipe / "model.pt"
    folder = FINAL_DIR / recipe
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    folder.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(folder / "refit_history.csv", index=False)
    refit = {"initialization": "original_pretrained_dinov2_fresh_heads",
             "data": "all_training_csv_rows", "images_per_epoch": len(train_full),
             "epochs": epochs, "cosine_t_max": EPOCHS}
    metadata = {"settings": settings, "recipe": recipe, "unfrozen_blocks": depth,
                "seen_pairs": train_pairs.tolist(), "selection": selection["selection"],
                "gamma": selection["gamma"], "refit": refit}
    atomic_save(checkpoint, {"state_dict": model.cpu().state_dict(), **metadata})
    lock = {**metadata, "selection_plan": plan, "checkpoint_sha256": sha256_file(checkpoint),
            "checkpoint_bytes": checkpoint.stat().st_size}
    atomic_json(folder / "selection_lock.json", lock)
    return lock


def refit_full_train():
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development.")
    if json.loads((DEVELOPMENT_DIR / "selection_plan.json").read_text(encoding="utf-8")) != plan:
        raise RuntimeError("The development plan changed after selection.")
    completed = {}
    for recipe, depth in RECIPES.items():
        if (FINAL_DIR / recipe / "selection_lock.json").is_file():
            completed[recipe] = read_final_lock(recipe)
        else:
            completed[recipe] = refit_recipe(recipe, depth, plan["recipes"][recipe])
    return completed


runs = {} if DEBUG else refit_full_train()


# 5. Evaluation

Verify all four locks and checkpoints. Add ten official pair identities using only the crop and disease columns and sort them by concept IDs, after the 323 training-derived pairs. Read only filenames for the image loaders. Each model generates both raw and calibrated predictions before per-image Ground Truth is joined. Report Top-1 and Top-5 for seen images, unseen images and all images over the common 333 candidates, plus the Top-1 harmonic mean. Restricted sets and the true-crop score are diagnostic.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    mean, std = (torch.tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["imagenet"])
    parts, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        crop_logits, disease_logits = model(images.to(DEVICE))
        scores = pair_scores(crop_logits, disease_logits, pairs).cpu()
        if perturb is None:
            concepts.append(torch.stack((crop_logits.argmax(1), disease_logits.argmax(1)), dim=1).cpu())
        if scores.shape != (len(images), len(pairs)) or not torch.isfinite(scores).all():
            raise ValueError("Unexpected official candidate predictions.")
        parts.append(scores)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(parts), torch.cat(concepts) if concepts else None



def predict_recipe(recipe, run, pairs, filenames):
    saved = torch.load(CHECKPOINT_DIR / recipe / "model.pt", map_location="cpu", weights_only=True)
    for key in ("settings", "recipe", "unfrozen_blocks", "selection", "gamma", "seen_pairs", "refit"):
        if saved[key] != run[key]:
            raise RuntimeError(f"Checkpoint and selection lock differ for {recipe}: {key}.")
    model = MultiTaskNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model = model.to(DEVICE).eval()
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    for rule, gamma in (("raw", 0.0), ("calibrated", run["gamma"])):
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    return predictions, clean, heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, clean, (seen_heads, unseen_heads) = forward_pass
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    same_crop_candidates = pairs[:, 0].eq(pairs[y_unseen, :1])
    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    for name, prediction in predictions.items():
        gamma = prediction["gamma"]
        raw_scores = clean[1]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        final_margins = raw_margins + gamma
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"scoring": name, "gamma": gamma, "unfrozen_blocks": run["unfrozen_blocks"],
                   "selected_on_tune": run["recipe"] == plan["recipe_selected_on_tune"],
                   "epoch": run["selection"]["epoch"], "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"scoring": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"scoring": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"scoring": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"scoring": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def depth_comparisons(rows):
    control = {row["scoring"]: row for row in rows if row["unfrozen_blocks"] == 0}
    output = []
    for row in rows:
        if row["unfrozen_blocks"] == 0:
            continue
        ref = control[row["scoring"]]
        gains = {f"{metric}_gain_pp": round(row[metric] - ref[metric], 2) for metric in
                 ("unseen_top1", "unseen_top5", "harmonic_mean")}
        seen_change = round(row["seen_top1"] - ref["seen_top1"], 2)
        gap = row["abs_seen_unseen_gap_pp"]
        output.append({"recipe": row["recipe"], "unfrozen_blocks": row["unfrozen_blocks"],
                       "scoring": row["scoring"], "reference": ref["recipe"], **gains,
                       "seen_top1_change_pp": seen_change, "seen_unseen_gap_pp": gap,
                       "all_three_increased": min(gains.values()) > 0,
                       "meets_balance_goal": min(gains.values()) > 0 and seen_change >= -2. and gap <= 10.})
    return output


def official_evaluation():
    if DEBUG or not runs or any(read_final_lock(recipe) != runs[recipe] for recipe in RECIPES):
        raise RuntimeError("Complete development and refit for every depth before official evaluation.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    if set(filenames[0]) & set(filenames[1]) or set(train_full.image) & (set(filenames[0]) | set(filenames[1])):
        raise ValueError("Training and official split filenames overlap.")
    predictions = {recipe: predict_recipe(recipe, runs[recipe], pairs, filenames) for recipe in RECIPES}
    # No per-image labels are retrieved until every locked rule has predicted.
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    summary = {"selection": runs, "recipe_selected_on_tune": plan["recipe_selected_on_tune"],
               "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    winners = {}
    for recipe, prediction in predictions.items():
        report, seen_winners = score_trial(runs[recipe], pairs, prediction, y_seen, y_unseen)
        winners.update({f"{recipe}_{rule}": values for rule, values in seen_winners.items()})
        for key, values in report.items():
            for row in values:
                row["recipe"] = recipe
            summary.setdefault(key, []).extend(values)
    summary["depth_gains"] = depth_comparisons(summary["results"])
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()


# 6. Save

- `results/approach2_a2_finetune_depth/<run_id>/development/` contains per-epoch selection, calibration grids and the independent check.
- `results/approach2_a2_finetune_depth/<run_id>/final/` contains each recipe's refit history and lock.
- `results/approach2_a2_finetune_depth/<run_id>/official/pair_identity_order/` contains eight raw/calibrated rows in `official_comparison.csv`, gains versus frozen A2 in `depth_gains.csv`, and ranking, robustness and confusion diagnostics.
- `checkpoints/approach2_a2_finetune_depth/<run_id>/<recipe>/model.pt` contains each complete DINOv2 and both heads.

In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    for name in ("depth_gains", "restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['recipe']}_{row['scoring']}.json", row)
    atomic_json(OFFICIAL_DIR / "official_report.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    display(table[["recipe", "unfrozen_blocks", "scoring", "gamma", "epoch", "selected_on_tune",
                   "seen_top1", "unseen_top1", "unseen_top5", "overall_top1", "harmonic_mean"]])
    display(pd.DataFrame(evaluation[0]["depth_gains"]))
    for recipe in RECIPES:
        print(recipe, "checkpoint:", (CHECKPOINT_DIR / recipe / "model.pt").resolve())
    print("Official results:", OFFICIAL_DIR.resolve())
else:
    print("Debug logs:", DEVELOPMENT_DIR.resolve())
